# SE446 — Big Data Engineering | Week 10 Lab A
## Apache Kafka: Brokers, Topics, Producers, and Consumers

**Course:** SE446 Big Data Engineering — Alfaisal University  
**Instructor:** Prof. Anis Koubaa  
**Dataset:** Chicago Crimes (used throughout the course)

---

### What you will learn in this lab

| Part | Concept | Why it matters |
|------|---------|---------------|
| 1 | Broker connection | Understanding how clients discover the cluster |
| 2 | Topics & Partitions | The unit of organization and parallelism |
| 3 | Producers & Keys | How data enters Kafka and how ordering is guaranteed |
| 4 | Consumers & Offsets | How data is read — the pull model |
| 5 | Log retention | Why Kafka is NOT a queue — messages survive reads |
| 6 | Consumer Groups | Scaling consumption horizontally |
| 7 | Offsets deep-dive | Bookmarks, commits, and crash recovery |

> **Important:** Run cells in order. Practice cells are wrapped in `try/except` and will never block execution.

## Part 0: Setup

We need the `confluent-kafka` Python client — the official Kafka library maintained by Confluent. This is the same library used in production systems.

In [2]:
!pip install confluent-kafka -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 49.1 MB/s eta 0:00:00


### Environment Configuration

The cell below auto-detects whether you are running on **Google Colab** or **locally**.

- **Colab** connects to an external broker on port 9093 (the advertised listener for external clients).
- **Local** connects to `localhost:9092` (assumes you have an SSH tunnel or are running on the cluster).

**Replace `YOUR_NAME`** with your actual name or student ID — this ensures your topics do not collide with other students on the shared cluster.

In [1]:
STUDENT_ID = "akoubaa"   # <-- CHANGE THIS to your name or student ID

import os
import json
import uuid
import time

ON_COLAB = "COLAB_RELEASE_TAG" in os.environ

if ON_COLAB:
    BROKER = "134.209.172.50:9093"
    print("Environment: Google Colab (external listener)")
else:
    BROKER = "localhost:9092"
    print("Environment: Local (SSH tunnel or on-cluster)")

TOPIC = f"crimes-{STUDENT_ID}"

print(f"Broker:     {BROKER}")
print(f"Topic:      {TOPIC}")
print(f"Student ID: {STUDENT_ID}")

Environment: Google Colab (external listener)
Broker:     134.209.172.50:9093
Topic:      crimes-akoubaa
Student ID: akoubaa


---
## Part 1: Connecting to Kafka (Concept: Broker)

### What is a Kafka Broker?

A **broker** is a single Kafka server. It does three things:

1. **Stores data** on disk in an append-only log (not in memory like Redis).
2. **Serves clients** — producers send data to brokers, consumers read data from brokers.
3. **Coordinates the cluster** — brokers share metadata so any broker can tell a client where to find a specific partition.

When we connect, the broker returns **cluster metadata**: the list of all brokers, all topics, and which broker hosts which partition. This is how Kafka clients discover the full cluster from a single bootstrap address.

Think of it like calling a company's main phone number — the receptionist (bootstrap broker) tells you which department (broker) handles your request (partition).

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

# Connect to the broker using AdminClient
admin = AdminClient({"bootstrap.servers": BROKER})

# List cluster metadata
metadata = admin.list_topics(timeout=10)

print("=== Cluster Info ===")
print(f"Cluster ID:       {metadata.cluster_id}")
print(f"Controller broker: {metadata.controller_id}")
print(f"Number of brokers: {len(metadata.brokers)}")
print()

print("=== Brokers ===")
for broker_id, broker in metadata.brokers.items():
    print(f"  Broker {broker_id}: {broker}")

print(f"\n=== Topics ({len(metadata.topics)} total) ===")
for topic_name in sorted(metadata.topics.keys())[:15]:
    print(f"  {topic_name}")
if len(metadata.topics) > 15:
    print(f"  ... and {len(metadata.topics) - 15} more")

### Comprehension Check

**Q1: What information did the broker return when we connected? Why does the broker need to advertise its address to external clients?**

**Q2: We only provided ONE broker address in `bootstrap.servers`. How does the client learn about ALL brokers in the cluster?**

> *Hint: Think about what "bootstrap" means — it is the starting point, not the only connection.*

---
## Part 2: Topics and Partitions (Concept: Topics, Partitions)

### What are Topics and Partitions?

A **topic** is a named category for messages — like a table in a database or a folder in a file system. All crime events go to the `crimes` topic; all alerts go to an `alerts` topic.

A **partition** is the unit of parallelism inside a topic. When you create a topic with 3 partitions, the data is split across 3 independent logs:

```
Topic: crimes-student1
├── Partition 0: [msg0, msg3, msg6, ...]
├── Partition 1: [msg1, msg4, msg7, ...]
└── Partition 2: [msg2, msg5, msg8, ...]
```

**Why does this matter?**
- Each partition can be read by a **different consumer in parallel**.
- In Week 11, when we connect Spark Streaming to Kafka, **3 partitions = 3 parallel Spark tasks** reading simultaneously.
- More partitions = more parallelism, but also more overhead. Choose wisely.

In [ ]:
# Create our topic with 3 partitions
new_topic = NewTopic(TOPIC, num_partitions=3, replication_factor=1)
futures = admin.create_topics([new_topic])

for topic_name, future in futures.items():
    try:
        future.result()  # Block until topic is created
        print(f"Topic '{topic_name}' created successfully!")
    except Exception as e:
        print(f"Topic '{topic_name}': {e}")

# Describe the topic
metadata = admin.list_topics(topic=TOPIC, timeout=10)
topic_meta = metadata.topics[TOPIC]
print(f"\n=== Topic: {TOPIC} ===")
print(f"Partitions: {len(topic_meta.partitions)}")
for pid, pmeta in sorted(topic_meta.partitions.items()):
    print(f"  Partition {pid}: leader=broker {pmeta.leader}, replicas={pmeta.replicas}")

### Comprehension Check

**Q: Why did we choose 3 partitions? What would happen if we chose 1? What about 100?**

> *Think about: parallelism, resource overhead, and the number of consumers that can read simultaneously.*

### Practice: Create a Second Topic

Create a topic called `alerts-{STUDENT_ID}` with **2 partitions**. This simulates a real scenario where different event types go to different topics.

In [ ]:
# PRACTICE: Create a topic called f"alerts-{STUDENT_ID}" with 2 partitions
# Hint: Use NewTopic() and admin.create_topics()
try:
    # YOUR CODE HERE
    practice_topic = NewTopic(f"alerts-{STUDENT_ID}", num_partitions=2, replication_factor=1)  # Example
    pass  # Replace this with your code
except Exception as e:
    print(f"Practice skipped or error: {e}")

---
## Part 3: Producing Messages with Keys (Concept: Producers, Hash Partitioning)

### What does a Producer do?

A **producer** sends messages to a Kafka topic. Each message has:
- A **value** (the actual data — e.g., a JSON crime event)
- An optional **key** (used to control which partition receives the message)
- A **topic** (where to send it)

### Why do keys matter?

When you provide a key, Kafka uses **hash partitioning**: `hash(key) % num_partitions` determines the partition. This guarantees that **all messages with the same key always go to the same partition**, which means they are **ordered relative to each other**.

For our crime data, we use the **district number** as the key. This means all crimes from District 8 always land in the same partition and stay in order.

### What is `flush()` and why is it critical?

`producer.produce()` is **asynchronous** — it puts the message in an internal buffer but does NOT send it immediately. The `flush()` call blocks until all buffered messages have been sent to the broker. **Without flush(), your messages may never be delivered** — especially in a notebook where the Python process can exit before the buffer is flushed.

In [ ]:
from confluent_kafka import Producer

producer = Producer({"bootstrap.servers": BROKER})

# Delivery callback — called once per message after broker confirms receipt
def delivery_report(err, msg):
    if err:
        print(f"  FAILED: {err}")
    else:
        print(f"  Delivered: partition={msg.partition()}, offset={msg.offset()}, key={msg.key().decode()}")

# 5 sample crime events with district as key
crimes = [
    {"case_id": "JE100001", "date": "2024-01-15", "type": "THEFT",    "district": 8,  "arrest": False, "lat": 41.87, "lon": -87.62},
    {"case_id": "JE100002", "date": "2024-01-15", "type": "BATTERY",  "district": 11, "arrest": True,  "lat": 41.86, "lon": -87.66},
    {"case_id": "JE100003", "date": "2024-01-16", "type": "BURGLARY", "district": 8,  "arrest": False, "lat": 41.88, "lon": -87.63},
    {"case_id": "JE100004", "date": "2024-01-16", "type": "ASSAULT",  "district": 25, "arrest": True,  "lat": 41.91, "lon": -87.75},
    {"case_id": "JE100005", "date": "2024-01-17", "type": "THEFT",    "district": 11, "arrest": False, "lat": 41.85, "lon": -87.67},
]

print(f"Producing 5 crime events to topic '{TOPIC}' with district as key...\n")
for crime in crimes:
    key = str(crime["district"])
    value = json.dumps(crime)
    producer.produce(TOPIC, key=key, value=value, callback=delivery_report)

# flush() blocks until ALL buffered messages are delivered
remaining = producer.flush(timeout=10)
print(f"\nFlush complete. Messages still in buffer: {remaining}")

### Observe the Output

Look at the delivery confirmations above. Notice:

- Messages with **key "8"** (District 8) all went to the **same partition**.
- Messages with **key "11"** (District 11) all went to the **same partition**.
- The partition numbers are determined by `hash("8") % 3` and `hash("11") % 3`.

This is **hash partitioning** in action — it guarantees ordering per entity (district) without any coordination.

### Comprehension Check

**Q1: What would happen if you produce messages WITHOUT a key? How would Kafka decide which partition to use?**

**Q2: Imagine District 8 generates 80% of all crime reports. Is hash partitioning still a good idea? What problem does this create?**

> *Hint: Think about "data skew" — one partition getting much more data than others.*

### Practice: Produce Without Keys

Produce 3 messages **without** a key and observe the partition assignment. Without a key, Kafka uses **round-robin** (or sticky partitioning) — messages spread evenly across partitions.

In [ ]:
# PRACTICE: Produce 3 messages WITHOUT a key and observe which partitions they land on
# What distribution pattern do you expect?
# Hint: Use producer.produce(TOPIC, value=..., callback=delivery_report) — no key parameter
try:
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Practice skipped or error: {e}")

---
## Part 4: Consuming Messages (Concept: Consumers, Offsets, Pull Model)

### How does a Consumer work?

Unlike traditional messaging systems that **push** messages to consumers, Kafka uses a **pull model**: the consumer calls `poll()` to ask the broker for new messages. This has several advantages:

1. **Backpressure control** — a slow consumer does not get overwhelmed; it reads at its own pace.
2. **Batch efficiency** — the consumer can pull many messages at once.
3. **Replayability** — the consumer controls its position (offset), not the broker.

### Key configuration:
- **`group.id`** — identifies which consumer group this consumer belongs to. Consumers in the same group share the work; consumers in different groups each get all messages.
- **`auto.offset.reset`** — what to do when there is no committed offset:
  - `earliest` = start from the beginning (read all historical data)
  - `latest` = start from now (skip old data, only read new messages)
- **Offsets** — each message in a partition has a sequential number (0, 1, 2, ...). The consumer tracks which offset it has read up to.

In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    "bootstrap.servers": BROKER,
    "group.id": f"{STUDENT_ID}-lab-group",
    "auto.offset.reset": "earliest",       # Start from the very first message
    "enable.auto.commit": True,
})

consumer.subscribe([TOPIC])

print(f"Consuming from topic '{TOPIC}'...\n")
print(f"{'Partition':<12} {'Offset':<10} {'Key':<10} {'Value'}")
print("-" * 80)

msg_count = 0
empty_polls = 0

while empty_polls < 10:  # Stop after 10 consecutive empty polls
    msg = consumer.poll(timeout=1.0)
    if msg is None:
        empty_polls += 1
        continue
    if msg.error():
        print(f"Consumer error: {msg.error()}")
        continue
    empty_polls = 0
    msg_count += 1
    key = msg.key().decode() if msg.key() else "None"
    value = msg.value().decode()
    print(f"{msg.partition():<12} {msg.offset():<10} {key:<10} {value[:60]}")

print(f"\nTotal messages consumed: {msg_count}")
consumer.close()

### Understanding the Output

Notice that the messages may **not** appear in the order they were produced. Why?

- Kafka guarantees ordering **within a partition**, not across partitions.
- The consumer reads from partition 0, then partition 1, then partition 2 (or in whatever order the broker assigns).
- Within each partition, the offsets are sequential (0, 1, 2, ...).

This is a fundamental trade-off: **more partitions = more parallelism, but no global ordering**.

### Comprehension Check

**Q: Why does Kafka use a PULL model (consumer.poll()) instead of pushing messages to consumers? What is the advantage for a slow consumer processing complex analytics?**

> *Think about what happens when the consumer is slower than the producer. In a push model, the consumer gets overwhelmed. In a pull model...*

---
## Part 5: Kafka Retains Messages (Concept: Log vs Queue)

### This is the KEY difference between Kafka and traditional message queues

In **RabbitMQ** (a traditional queue):
- Consumer reads a message → message is **deleted** from the queue.
- Once consumed, the data is gone forever.
- If you want two systems to read the same data, you need two queues.

In **Kafka** (a distributed log):
- Consumer reads a message → message **stays in the log**.
- Messages are retained for a configurable period (default: **7 days**).
- Multiple consumers (or consumer groups) can read the same data independently.
- A consumer can **replay** old data by resetting its offset.

This is why Kafka is called a **distributed commit log**, not a message queue. It is more like a database table that you append to and read from, than a queue that you drain.

### Why does this matter?
- **Crash recovery**: A consumer crashes → restart it, read from where it left off.
- **Replay**: Need to reprocess data with a bug fix? Reset the offset and re-read.
- **Multiple consumers**: Analytics team AND alerting team can both read the same crime data.

Let us prove this by reading the same messages again with a **brand new consumer group**.

In [ ]:
# Create a BRAND NEW consumer group — never seen before
new_group_id = f"{STUDENT_ID}-replay-{uuid.uuid4().hex[:6]}"

replay_consumer = Consumer({
    "bootstrap.servers": BROKER,
    "group.id": new_group_id,
    "auto.offset.reset": "earliest",
})

replay_consumer.subscribe([TOPIC])

print(f"New consumer group: {new_group_id}")
print(f"Reading ALL messages from '{TOPIC}' again...\n")

msg_count = 0
empty_polls = 0

while empty_polls < 10:
    msg = replay_consumer.poll(timeout=1.0)
    if msg is None:
        empty_polls += 1
        continue
    if msg.error():
        continue
    empty_polls = 0
    msg_count += 1

print(f"Messages read by NEW consumer group: {msg_count}")
print("The data was NOT deleted after Part 4 consumed it!")
replay_consumer.close()

### What just happened?

We created a completely new consumer group that had **never read from this topic before**. Because `auto.offset.reset=earliest`, it started from offset 0 and read **all** the messages — the same ones we already consumed in Part 4.

In RabbitMQ, those messages would be gone. In Kafka, they are still there.

### Comprehension Check

**Q: A new data science team joins your project 2 weeks after launch. They need all historical crime data to train a model. How can they access it without building a new pipeline or re-ingesting data?**

> *Hint: They just need to create a new...*

---
## Part 6: Consumer Groups (Concept: Parallel Consumption)

### Why do Consumer Groups exist?

Imagine your crime data topic receives 100,000 messages per second. A single consumer can only process 10,000/sec. What happens?

The **consumer lag** — the gap between the latest message and the consumer's current position — grows forever. The consumer falls further and further behind.

The solution: **consumer groups**. Within a consumer group, Kafka **distributes partitions** among the consumers:

```
Topic: crimes (3 partitions)

Consumer Group A (analytics):
  Consumer 1 → reads Partition 0
  Consumer 2 → reads Partition 1
  Consumer 3 → reads Partition 2

Consumer Group B (alerting):
  Consumer 1 → reads ALL partitions (only 1 consumer in this group)
```

**Key rules:**
- Within a group: each partition is assigned to exactly ONE consumer.
- Across groups: each group independently reads ALL messages.
- If you have more consumers than partitions, the extras sit idle.

In [ ]:
# Show topic info: partition count and message distribution
metadata = admin.list_topics(topic=TOPIC, timeout=10)
topic_meta = metadata.topics[TOPIC]

print(f"=== Topic: {TOPIC} ===")
print(f"Number of partitions: {len(topic_meta.partitions)}\n")

# Show watermark offsets for each partition
# Watermarks tell us: low = earliest available offset, high = next offset to be written
temp_consumer = Consumer({
    "bootstrap.servers": BROKER,
    "group.id": f"_watermark-check-{uuid.uuid4().hex[:6]}",
})

total_messages = 0
print(f"{'Partition':<12} {'Low Offset':<14} {'High Offset':<14} {'Messages'}")
print("-" * 55)

for pid in sorted(topic_meta.partitions.keys()):
    from confluent_kafka import TopicPartition
    low, high = temp_consumer.get_watermark_offsets(
        TopicPartition(TOPIC, pid), timeout=10
    )
    count = high - low
    total_messages += count
    print(f"{pid:<12} {low:<14} {high:<14} {count}")

print(f"\nTotal messages across all partitions: {total_messages}")
temp_consumer.close()

### Comprehension Check

**Q: If your topic has 3 partitions and your consumer group has 5 consumers, what happens to the extra 2 consumers? Why is this a waste of resources?**

> *Key insight: The maximum useful parallelism equals the number of partitions.*

### Practice: Read with a Different Consumer Group

Create a consumer with a different `group.id` and read all messages. Since this is a new group, it should receive ALL messages independently (just like the analytics and alerting teams in the diagram above).

In [ ]:
# PRACTICE: Create a consumer with group.id = f"{STUDENT_ID}-dashboard-group"
# Read all messages. Does it get the same messages as the first consumer?
# Hint: Use auto.offset.reset="earliest" and poll() in a loop
try:
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Practice skipped or error: {e}")

---
## Part 7: Exploring Offsets (Concept: Offsets as Bookmarks)

### What are offsets?

Every message in a partition has a **sequential offset** — a monotonically increasing integer starting from 0.

```
Partition 0:  [offset 0] [offset 1] [offset 2] [offset 3] ...
Partition 1:  [offset 0] [offset 1] [offset 2] ...
Partition 2:  [offset 0] [offset 1] ...
```

The consumer's **committed offset** is its bookmark — the position where it will resume reading after a restart.

### Watermark offsets

Each partition has two watermarks:
- **Low watermark** — the earliest offset still available (messages before this have been deleted by retention policy).
- **High watermark** — the offset of the NEXT message to be written (i.e., one past the last message).

So the number of available messages = `high - low`.

In [ ]:
from confluent_kafka import TopicPartition

# Use a consumer to inspect offsets in detail
offset_consumer = Consumer({
    "bootstrap.servers": BROKER,
    "group.id": f"_offset-inspector-{uuid.uuid4().hex[:6]}",
})

metadata = admin.list_topics(topic=TOPIC, timeout=10)
partitions = sorted(metadata.topics[TOPIC].partitions.keys())

print(f"=== Offset Details for '{TOPIC}' ===\n")
print(f"{'Partition':<12} {'Low (earliest)':<18} {'High (next write)':<20} {'Available msgs'}")
print("-" * 65)

for pid in partitions:
    tp = TopicPartition(TOPIC, pid)
    low, high = offset_consumer.get_watermark_offsets(tp, timeout=10)
    available = high - low
    print(f"{pid:<12} {low:<18} {high:<20} {available}")

print("\nInterpretation:")
print("  - Low offset = earliest message still available (not yet deleted by retention)")
print("  - High offset = next offset to be assigned to a new message")
print("  - If low > 0, older messages have been removed by retention policy")

offset_consumer.close()

### Comprehension Check

**Q: If a consumer crashes after reading offset 4 but BEFORE committing, what offset will it restart from? What does this mean for message delivery — could some messages be processed twice?**

> *This is the "at-least-once" delivery guarantee. Kafka guarantees no message loss, but duplicates are possible if the consumer crashes between processing and committing. This is why idempotent processing matters.*

---
## Part 8: Practice Challenges

Three independent exercises to reinforce what you have learned. Each is wrapped in `try/except` so they do not block the rest of the notebook.

### Challenge 1: Custom Producer

Write a producer that sends **10 weather sensor readings** to a new topic. Each reading should include: `sensor_id`, `temperature`, `humidity`, and `timestamp`. Use `sensor_id` as the key so readings from the same sensor stay ordered in the same partition.

In [ ]:
# CHALLENGE 1: Write a producer that sends 10 weather sensor readings
# Each reading should have: sensor_id, temperature, humidity, timestamp
# Use sensor_id as the key so readings from the same sensor stay ordered
#
# Hints:
#   - Create a topic f"weather-{STUDENT_ID}" first (or reuse the crimes topic)
#   - Use json.dumps() to serialize the value
#   - Use producer.produce(topic, key=sensor_id, value=json_str, callback=delivery_report)
#   - Don't forget producer.flush()!
try:
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Challenge 1 skipped: {e}")

### Challenge 2: Filtered Consumer

Write a consumer that reads from the crimes topic but **only prints messages where `arrest` is `True`**. This simulates a real-world pattern where a consumer filters a stream for specific events.

In [ ]:
# CHALLENGE 2: Write a consumer that only prints messages where arrest=True
# Hint: Parse the JSON value with json.loads(), check the "arrest" field
#
# Steps:
#   1. Create a Consumer with a new group.id and auto.offset.reset="earliest"
#   2. Subscribe to TOPIC
#   3. Poll in a loop, parse JSON, filter for arrest==True
#   4. Don't forget to close the consumer!
try:
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Challenge 2 skipped: {e}")

### Challenge 3: Message Counter per Partition

Write a consumer that counts how many messages are in each partition and prints a summary. This helps you understand data distribution and detect potential skew.

In [ ]:
# CHALLENGE 3: Write a consumer that counts how many messages are in each partition
# Print a summary like: "Partition 0: 5 messages, Partition 1: 3 messages, ..."
#
# Hints:
#   - Use a dictionary: partition_counts = {}
#   - msg.partition() gives you the partition number
#   - Use a new group.id with auto.offset.reset="earliest"
try:
    # YOUR CODE HERE
    pass
except Exception as e:
    print(f"Challenge 3 skipped: {e}")

---
## Part 9: Cleanup

Run this cell to delete the test topics you created during the lab. **Uncomment the lines** when you are ready to clean up.

In [ ]:
# Uncomment the lines below to delete your test topics
# This is good practice to avoid leaving orphan topics on the shared cluster

topics_to_delete = [
    # TOPIC,
    # f"alerts-{STUDENT_ID}",
    # f"weather-{STUDENT_ID}",
]

if topics_to_delete:
    futures = admin.delete_topics(topics_to_delete)
    for topic_name, future in futures.items():
        try:
            future.result()
            print(f"Deleted: {topic_name}")
        except Exception as e:
            print(f"Could not delete {topic_name}: {e}")
else:
    print("No topics to delete. Uncomment the topic names above to clean up.")

---
## Part 10: Assessment Preparation

After completing this lab, you should be able to answer the following questions confidently. These cover the core Kafka concepts tested in SE446.

---

**1. What is the difference between a Kafka topic and a partition?**

> A topic is a named category (like "crimes"). A partition is a subdivision of a topic — the unit of parallelism and ordering. A topic with 3 partitions has 3 independent, ordered logs.

**2. Why does Kafka use hash partitioning when a key is provided?**

> Hash partitioning ensures all messages with the same key go to the same partition. This guarantees ordering for that key (e.g., all District 8 crimes arrive in order) without requiring coordination.

**3. What happens to a message after a consumer reads it? How is this different from RabbitMQ?**

> In Kafka, the message stays in the log (retained for 7 days by default). In RabbitMQ, the message is deleted after acknowledgment. Kafka's model enables replay, multiple consumers, and crash recovery.

**4. Explain the difference between `auto.offset.reset=earliest` and `auto.offset.reset=latest`.**

> `earliest` = start reading from the very first available message (offset 0 or the low watermark). `latest` = start reading only new messages produced after the consumer starts. This only applies when there is no committed offset for the group.

**5. A Kafka topic has 3 partitions and your consumer group has 4 consumers. Describe what happens.**

> Three consumers each get one partition. The fourth consumer sits idle — it has no partition to read. The maximum useful parallelism equals the number of partitions.

**6. Your Spark Streaming job reads from a Kafka topic with 6 partitions. How many Spark tasks will read in parallel?**

> Six. Spark creates one task per Kafka partition, so 6 partitions = 6 parallel reading tasks. This is why partition count directly affects Spark parallelism.

**7. What does `producer.flush()` do? What could go wrong without it?**

> `flush()` blocks until all buffered messages are delivered to the broker. Without it, `produce()` only puts messages in an internal buffer — they may never be sent if the program exits before the buffer is flushed.

**8. A consumer crashes after reading offset 5 but before committing. What happens when it restarts?**

> It restarts from the last committed offset (e.g., offset 3). Messages 3, 4, and 5 will be reprocessed. This is "at-least-once" delivery — no data loss, but possible duplicates.

**9. Your team needs to process crime data AND send alerts from the same topic. How would you set this up with consumer groups?**

> Create two consumer groups: one for processing (e.g., "analytics-group") and one for alerting (e.g., "alerts-group"). Each group independently reads ALL messages from the topic. No data duplication in Kafka — both groups read the same log.

**10. Why is Kafka described as a "distributed commit log" rather than a "message queue"?**

> A message queue deletes messages after consumption. Kafka retains messages in an append-only log regardless of how many consumers read them. Messages have durable, sequential offsets. Consumers can replay, seek, and re-read — just like reading a database log. This makes Kafka suitable for event sourcing, stream processing, and data integration beyond simple point-to-point messaging.

---

**End of Lab A.** In Lab B, we will connect Spark Structured Streaming to this Kafka topic and process crime data in real time.